# 03 — Deep Learning para radiografías: Sana / Neumonía / COVID-19

Este notebook documenta el módulo `ml-inference`: dataset, limpieza de `masks/`, preprocesamiento, entrenamiento, comparación CNN/ResNet18/EfficientNet-B0, métricas, matrices de confusión, criterio clínico e integración con dashboard.

In [ ]:

from pathlib import Path
import json
import os
import subprocess
import textwrap
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start: Path | None = None) -> Path:
    """Busca la raíz del proyecto localizando docker-compose.yml."""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "docker-compose.yml").exists():
            return candidate
    # Si el notebook se ejecuta desde notebooks/ antes de abrir el repo correctamente.
    return start


ROOT = find_project_root()
print(f"ROOT = {ROOT}")


def read_json(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}")
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_env(path: Path) -> dict:
    env = {}
    path = Path(path)
    if not path.exists():
        return env
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env


def run_cmd(cmd: str, timeout: int = 60) -> str:
    """Ejecuta un comando shell y devuelve stdout/stderr como texto."""
    print(f"$ {cmd}")
    try:
        result = subprocess.run(
            cmd,
            shell=True,
            cwd=ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        print(f"exit_code={result.returncode}")
        return (result.stdout or "") + (result.stderr or "")
    except Exception as exc:
        print(f"[error] {exc}")
        return str(exc)


def show_bar(series, title: str, ylabel: str = "valor"):
    ax = pd.Series(series).plot(kind="bar")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


print("=== Dataset de radiografías preparado ===")

subset_dir = ROOT / "data" / "covid-subset"
train_csv = subset_dir / "train.csv"
val_csv = subset_dir / "val.csv"
test_csv = subset_dir / "test.csv"
metadata_path = subset_dir / "metadata.json"

print(f"subset_dir existe: {subset_dir.exists()}")
print(f"train.csv existe: {train_csv.exists()}")
print(f"val.csv existe: {val_csv.exists()}")
print(f"test.csv existe: {test_csv.exists()}")
print(f"metadata.json existe: {metadata_path.exists()}")

if metadata_path.exists():
    metadata = read_json(metadata_path, default={})
    print("\nmetadata.json:")
    print(json.dumps(metadata, indent=2, ensure_ascii=False))

if train_csv.exists() and val_csv.exists() and test_csv.exists():
    train = pd.read_csv(train_csv)
    val = pd.read_csv(val_csv)
    test = pd.read_csv(test_csv)

    print("\nTamaños:")
    print(f"train: {len(train)}")
    print(f"val:   {len(val)}")
    print(f"test:  {len(test)}")

    counts = pd.concat([
        train.assign(split="train"),
        val.assign(split="val"),
        test.assign(split="test"),
    ]).groupby(["split", "class_target"]).size().reset_index(name="count")

    display(counts)

    pivot = counts.pivot(index="class_target", columns="split", values="count").fillna(0)
    display(pivot)

    ax = pivot.plot(kind="bar")
    ax.set_title("Distribución de clases por split")
    ax.set_ylabel("Número de imágenes")
    ax.set_xlabel("Clase")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Verificación crítica: no deben aparecer masks/
    all_paths = pd.concat([train["filepath"], val["filepath"], test["filepath"]])
    masks_count = all_paths.str.contains("/masks/|\\\\masks\\\\", regex=True).sum()
    print(f"\nRutas con masks/: {masks_count}")

    if masks_count == 0:
        print("OK: el dataset usa solo carpetas images/.")
    else:
        print("ERROR: hay rutas de masks/ en los CSV.")


## 1. Dataset y clases

Dataset local:

```text
COVID-19_Radiography_Dataset/
├── COVID/images/
├── Normal/images/
├── Viral Pneumonia/images/
└── Lung_Opacity/images/
```

Mapeo final:

| Origen | Clase final |
|---|---|
| `Normal` | `Sana` |
| `Viral Pneumonia` | `Neumonía` |
| `Lung_Opacity` | `Neumonía` |
| `COVID` | `COVID-19` |

La versión corregida de `prepare_data.py` usa solo `images/` y excluye `masks/`.

In [ ]:
subset_dir = ROOT / "data/covid-subset"
metadata = read_json(subset_dir/"metadata.json", default={}) or {}
metadata


In [ ]:
# Cargar splits si existen
splits = {}
for split in ["train", "val", "test"]:
    path = subset_dir / f"{split}.csv"
    if path.exists():
        splits[split] = pd.read_csv(path)
        print(split, splits[split].shape)
    else:
        print(f"[missing] {path}")


In [ ]:
# Comprobar que no hay masks/ en los CSV
mask_rows = []
for split, df in splits.items():
    count = df["filepath"].astype(str).str.contains("/masks/|\\\\masks\\\\", regex=True).sum()
    mask_rows.append({"split": split, "filas_con_masks": int(count), "filas": len(df)})
pd.DataFrame(mask_rows)


In [ ]:
# Distribución por clase y split
rows = []
for split, df in splits.items():
    for cls, count in df["class_target"].value_counts().items():
        rows.append({"split": split, "class_target": cls, "count": count})
class_dist = pd.DataFrame(rows)
class_dist


In [ ]:
if not class_dist.empty:
    pivot = class_dist.pivot(index="class_target", columns="split", values="count").fillna(0)
    ax = pivot.plot(kind="bar")
    ax.set_title("Distribución de clases por split")
    ax.set_ylabel("nº imágenes")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


## 2. Visualización de ejemplos

Se muestran ejemplos por clase desde el CSV de entrenamiento. Si el dataset no está disponible localmente, esta celda simplemente avisará.

In [ ]:
from PIL import Image

if "train" in splits:
    train_df = splits["train"]
    for cls in sorted(train_df["class_target"].unique()):
        sample = train_df[train_df["class_target"] == cls].iloc[0]
        path = Path(sample["filepath"])
        # La ruta dentro del CSV suele ser /app/..., la convertimos a ruta local del repo si hace falta.
        local_path = path
        if str(path).startswith("/app/"):
            local_path = ROOT / str(path).replace("/app/", "")
        print(cls, local_path)
        if local_path.exists():
            img = Image.open(local_path)
            plt.figure(figsize=(4, 4))
            plt.imshow(img, cmap="gray")
            plt.title(f"{cls} — {sample['class_original']}")
            plt.axis("off")
            plt.show()
        else:
            print("No existe localmente:", local_path)


## 3. Preprocesamiento

Entrenamiento:

- resize a 256x256;
- `RandomResizedCrop(224)`;
- flip horizontal;
- rotación leve;
- ajuste leve de brillo/contraste;
- normalización ImageNet.

Validación/test/inferencia:

- resize directo a 224x224;
- tensor;
- normalización ImageNet.

No se aplican aumentos agresivos porque podrían alterar patrones radiológicos.

## 4. Modelos comparados

| Modelo | Tipo | Papel experimental |
|---|---|---|
| `simple_cnn` | CNN desde cero | Baseline. Sirve para medir cuánto aporta transfer learning. |
| `resnet18` | Transfer learning | Arquitectura clásica y robusta. |
| `efficientnet_b0` | Transfer learning eficiente | Mejor equilibrio coste/rendimiento. |

La selección final no se hace solo por accuracy, sino por comportamiento clínico: F1 macro, recall COVID y recall Neumonía.

In [ ]:
comparison_path = ROOT / "models/radiography/comparison/comparison.csv"
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
else:
    comparison = pd.DataFrame([
        {"version": "rx-efficientnetb0-20260515-2b92eec9", "backbone": "efficientnet_b0", "accuracy": 0.9600, "f1_macro": 0.9601, "recall_covid": 0.9800, "recall_neumonia": 0.9333},
        {"version": "rx-resnet18-20260515-d397f8e5", "backbone": "resnet18", "accuracy": 0.8933, "f1_macro": 0.8929, "recall_covid": 0.9667, "recall_neumonia": 0.8333},
        {"version": "rx-simplecnn-20260515-e8ac76f0", "backbone": "simple_cnn", "accuracy": 0.7022, "f1_macro": 0.7013, "recall_covid": 0.7000, "recall_neumonia": 0.7733},
    ])
comparison

print("=== Comparación CNN simple vs ResNet18 vs EfficientNet-B0 ===")

comparison_csv = ROOT / "models" / "radiography" / "comparison" / "comparison.csv"
comparison_md = ROOT / "models" / "radiography" / "comparison" / "comparison.md"

print(f"comparison.csv existe: {comparison_csv.exists()}")
print(f"comparison.md existe: {comparison_md.exists()}")

if comparison_csv.exists():
    comparison = pd.read_csv(comparison_csv)
    display(comparison)

    metric_cols = ["accuracy", "f1_macro", "recall_covid", "recall_neumonia"]
    for col in metric_cols:
        if col in comparison.columns:
            ax = comparison.set_index("backbone")[col].plot(kind="bar")
            ax.set_title(col)
            ax.set_ylim(0, 1)
            ax.set_ylabel("valor")
            ax.set_xlabel("modelo")
            plt.xticks(rotation=30, ha="right")
            plt.tight_layout()
            plt.show()

    best = comparison.sort_values(
        by=["f1_macro", "recall_covid", "recall_neumonia"],
        ascending=False,
    ).iloc[0]

    print("\nModelo seleccionado:")
    print(f"Versión: {best['version']}")
    print(f"Backbone: {best['backbone']}")
    print(f"Accuracy: {best['accuracy']:.4f}")
    print(f"F1 macro: {best['f1_macro']:.4f}")
    print(f"Recall COVID: {best['recall_covid']:.4f}")
    print(f"Recall Neumonía: {best['recall_neumonia']:.4f}")

if comparison_md.exists():
    print("\ncomparison.md:")
    print(comparison_md.read_text(encoding="utf-8"))


In [ ]:
metrics_cols = ["accuracy", "f1_macro", "recall_covid", "recall_neumonia"]
plot_df = comparison.set_index("backbone")[metrics_cols]
ax = plot_df.plot(kind="bar")
ax.set_title("Comparación de modelos de radiografías")
ax.set_ylabel("score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Gráficas individuales para lectura clara
for metric in ["accuracy", "f1_macro", "recall_covid", "recall_neumonia"]:
    ax = comparison.set_index("backbone")[metric].plot(kind="bar")
    ax.set_title(metric)
    ax.set_ylabel("score")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


## 5. Matrices de confusión

Se cargan las matrices de confusión generadas por `training.evaluate`.

In [ ]:
for png in sorted((ROOT/"models/radiography").glob("rx-*/confusion_matrix.png")):
    print(png.relative_to(ROOT))
    img = Image.open(png)
    plt.figure(figsize=(6, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.show()


from IPython.display import Image, display

print("=== Matrices de confusión generadas ===")

models_root = ROOT / "models" / "radiography"
artifact_dirs = sorted([p for p in models_root.iterdir() if p.is_dir() and p.name.startswith("rx-")])

for artifact in artifact_dirs:
    print(f"\n--- {artifact.name} ---")
    metadata = read_json(artifact / "metadata.json", default={})
    metrics = read_json(artifact / "metrics.json", default={})
    cm_path = artifact / "confusion_matrix.png"
    critical_path = artifact / "critical_analysis.md"

    print(f"Backbone: {metadata.get('config', {}).get('backbone', metadata.get('backbone', 'unknown'))}")
    if metrics:
        print(f"Accuracy: {metrics.get('accuracy')}")
        print(f"F1 macro: {metrics.get('f1_macro')}")
        print(f"Recall COVID: {metrics.get('recall_covid')}")
        print(f"Recall Neumonía: {metrics.get('recall_neumonia')}")

    if cm_path.exists():
        display(Image(filename=str(cm_path)))

    if critical_path.exists():
        print("\nAnálisis crítico:")
        print(critical_path.read_text(encoding="utf-8")[:2500])

## 6. Interpretación clínica

### Falso negativo COVID-19

Un COVID-19 clasificado como `Sana` o `Neumonía` es crítico porque puede implicar falta de aislamiento y contagio intrahospitalario.

### COVID-19 confundido con Neumonía

El tratamiento respiratorio puede compartir elementos, pero se pierde el componente epidemiológico.

### Neumonía clasificada como Sana

Puede retrasar tratamiento y seguimiento.

### Sana clasificada como patológica

Produce sobrecarga asistencial, pruebas innecesarias y ansiedad, aunque suele ser menos grave que un falso negativo clínico.

## 7. Selección del modelo

Con los resultados actuales:

- EfficientNet-B0 obtiene `accuracy = 0.9600`.
- EfficientNet-B0 obtiene `f1_macro = 0.9601`.
- EfficientNet-B0 obtiene `recall_covid = 0.9800`.
- EfficientNet-B0 obtiene `recall_neumonia = 0.9333`.

Por tanto, se selecciona **EfficientNet-B0** como modelo activo, porque ofrece mejor equilibrio entre rendimiento global y sensibilidad en enfermedades clínicamente relevantes.

## 8. Comandos usados

Preparación limpia:

```powershell
docker compose run --rm ml-inference python -m training.prepare_data `
  --raw-dir /app/COVID-19_Radiography_Dataset `
  --output-dir /app/data/covid-subset `
  --limit-per-class 1000 `
  --seed 42
```

Entrenamiento de modelos:

```powershell
docker compose run --rm ml-inference python -m training.train --backbone simple_cnn ...
docker compose run --rm ml-inference python -m training.train --backbone resnet18 ...
docker compose run --rm ml-inference python -m training.train --backbone efficientnet_b0 ...
```

Comparación:

```powershell
docker compose run --rm ml-inference python -m training.compare_models `
  --models-root /app/models/radiography `
  --output /app/models/radiography/comparison
```

## 9. Integración con dashboard

El dashboard llama internamente a:

```text
http://ml-inference:8001/predict
```

No usa `localhost` porque dentro del contenedor `dashboard`, localhost sería el propio dashboard.